In [16]:
!pip install datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import torch


In [17]:
train_df = pd.read_csv("/content/train_cleaned (1).txt", names=["text", "label"])


In [18]:
label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["label"])  # becomes 0, 1, 2


In [19]:
train_dataset = Dataset.from_pandas(train_df)
train_dataset = Dataset.from_pandas(train_df)


In [20]:
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

In [22]:
train_dataset = train_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/5167 [00:00<?, ? examples/s]

In [23]:
training_args = TrainingArguments(
    output_dir="./muril_finetuned",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=10,
    save_total_limit=1,
    remove_unused_columns=False,
)

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

In [25]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chaithrakaranth05 (chaithrakaranth05-manipal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,1.094600
20,1.067400
30,1.015900
40,0.975300
50,0.964000
60,0.891200
70,0.890400
80,0.797000
90,0.708800
100,0.759200


TrainOutput(global_step=1938, training_loss=0.19765086954515293, metrics={'train_runtime': 609.4701, 'train_samples_per_second': 25.434, 'train_steps_per_second': 3.18, 'total_flos': 1019630272050432.0, 'train_loss': 0.19765086954515293, 'epoch': 3.0})

In [26]:
test_df = pd.read_csv("/content/test3.txt", names=["text", "true_label"])

In [27]:
import torch

# Make sure model is on the same device as the inputs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Move inputs to the same device
inputs = tokenizer(
    test_df["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=500,
    return_tensors="pt"
).to(device)

# Predict
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=1)

# Convert predictions to numpy
pred_labels = preds.cpu().numpy()



In [28]:
print(type(preds))


<class 'torch.Tensor'>


In [29]:
test_df["predicted_label"] = label_encoder.inverse_transform(preds.cpu().numpy())

In [30]:
print(test_df[["text", "predicted_label"]])

                                                 text predicted_label
0   ಸಾಮ್ಯುಕ್ತಾ ಅವರ ಹೊಸ ಚಿತ್ರ 'ವಿದಾಮುಯರ್ಚಿ' ಬಾಕ್ಸ್ ...   entertainment
1   ಶ್ರೀಲೀನಾ-ಸಿದ್ಧಾರ್ಥ್ ರಾಜ್ ಶಾಂಡಿಲ್ಯ ಅವರ ಮುಂದಿನ ಚ...   entertainment
2   ವಿಶ್ವದ ದೊಡ್ಡ ಅನಕೊಂಡಾ ಪತ್ತೆ: ಕನಸಲ್ಲೂ ಬೆಚ್ಚಿಬೀಳೋ...          sports
3      'ರೈಡ್ 2' ಬಾಕ್ಸ್ ಆಫೀಸ್ ಕಲೆಕ್ಷನ್: ಮೊದಲ ದಿನದ ವರದಿ   entertainment
4   ಅಮೆಜಾನ್ ಸಮ್ಮರ್ ಸೇಲ್: ಸ್ಮಾರ್ಟ್‌ಫೋನ್‌ಗಳು ಕಡಿಮೆ ಬ...            tech
5              'ಅಯ್ಯನ ಮನೆ' ಓಟಿಟಿ ಬಿಡುಗಡೆ ದಿನಾಂಕ ಘೋಷಣೆ   entertainment
6   ಸ್ಯಾಮ್‌ಸಂಗ್ ಗ್ಯಾಲಕ್ಸಿ ಎಸ್24 ಅಲ್ಟ್ರಾ ಬಿಡುಗಡೆ: ಆ...            tech
7   'ಮೆಲೋಬ್ಬ ಮಾಯಾವಿ' ನಿರ್ಮಾಪಕ ಭರತ್ ಕುಮಾರ್ 43ನೇ ವಯಸ...   entertainment
8   ಅಭಿಷೇಕ ಶರ್ಮಾ ಕೇವಲ ಬೌಂಡರಿ ಸಿಕ್ಸರ್‌ಗಳಿಂದಲೇ 116 ರ...          sports
9   ಸಾಯಿ ರಾಜೇಶ್ ಬಾಬಿಲ್ ಸ್ಪಷ್ಟೀಕರಣದ ನಂತರ ಆಕ್ರೋಶ ವ್ಯ...   entertainment
10  'ಹಿಟ್ 3' ಚಿತ್ರದ ವಿಮರ್ಶೆ: ಪ್ರೇಕ್ಷಕರಿಂದ ಉತ್ತಮ ಪ್...   entertainment
11  ವಾಟ್ಸ್ಆಪ್‌ನಲ್ಲಿ ಅನೌನ್ ನಂಬರ್ನಿಂದ ಮೆಸೇಜ್ ಸಮಸ್ಯೆ ...            tech
12  ಐಪಿಎಲ್ 2025: ಪ್ಲೇಆಫ್ ಆಡಲಿರುವ 4 ತಂಡಗಳನ್ನು ಹೆಸರಿ...          sports
13  'ಕೆಜಿಎಫ್ ಚಾಪ್ಟರ್

In [31]:
from sklearn.metrics import accuracy_score

In [32]:
label_list = ['sports', 'tech', 'entertainment']
label_encoder = LabelEncoder()
label_encoder.fit(label_list)



LabelEncoder()

In [33]:
true_labels = label_encoder.transform(test_df["true_label"])

In [34]:
predicted_labels = preds.cpu().numpy()

In [35]:
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"\n✅ Accuracy on test set: {accuracy * 100:.2f}%")


✅ Accuracy on test set: 91.11%


In [36]:
from sklearn.metrics import classification_report

# Assuming y_test contains true labels and y_pred contains predicted labels
# Replace these with your actual variable names if different

# If you have label names (e.g., class names)
target_names = ['entertainment', 'sports', 'tech']

# Print the raw classification report
print(classification_report(true_labels, predicted_labels, target_names=target_names))



               precision    recall  f1-score   support

entertainment       0.90      1.00      0.95        18
       sports       0.80      0.80      0.80        10
         tech       1.00      0.88      0.94        17

     accuracy                           0.91        45
    macro avg       0.90      0.89      0.89        45
 weighted avg       0.92      0.91      0.91        45

